im2col函数

In [1]:


def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    N, C, H, W = input_data.shape
    #C是通道数，最初的通道数是3，（因为计算机RGB色彩空间只用红绿蓝三种颜色，就可以表示彩图，灰色图片只要一个通道；但是有的神经网络通道数目多达100多个，怎么理解呢？
    #类比理解为更多方面的信息，至于N是数量，H高，W宽不赘述
    out_h = (H + 2*pad - filter_h) // stride + 1
    out_w = (W + 2*pad - filter_w) // stride + 1
    #补全，卷积核滚过以后特征图高/宽公式
    img = np.pad(input_data, [(0, 0), (0, 0), (pad, pad), (pad, pad)], 'constant')
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))

    for y in range(filter_h):
       y_max = y + stride*out_h
    for x in range(filter_w):
       x_max = x + stride*out_w
       col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col

从下面开始运行

In [2]:
import numpy as np
class Convolution:
    def __init__(self, w, b, stride=1, pad=0):
        self.w = w
        self.b = b
        self.stride = stride
        self.pad = pad

    def forward(self, x):
        N, C, H, W = x.shape
        FN, C_w, HH, WW = self.w.shape
        assert C == C_w, "Input channel must match filter channel"

        H_out = (H + 2 * self.pad - HH) // self.stride + 1
        W_out = (W + 2 * self.pad - WW) // self.stride + 1

        x_col = im2col(x, HH, WW, self.stride, self.pad)
        w_col = self.w.reshape(FN, -1)

        out = x_col @ w_col.T + self.b
        out = out.reshape(N, H_out, W_out, FN).transpose(0, 3, 1, 2)

        return out



In [ ]:


def calc_conv_output_size(input_h, input_w, filter_h, filter_w, stride=1, pad=0):
    """计算卷积输出特征图的高和宽"""
    out_h = (input_h + 2 * pad - filter_h) // stride + 1
    out_w = (input_w + 2 * pad - filter_w) // stride + 1
    return out_h, out_w


def im2col(input_data, filter_h, filter_w, stride=1, pad=0):
    """
    把输入张量展开成二维矩阵，方便后续用矩阵乘法实现卷积

    input_data shape: (N, C, H, W)
    return shape: (N * out_h * out_w, C * filter_h * filter_w)
    """
    N, C, H, W = input_data.shape
    out_h, out_w = calc_conv_output_size(H, W, filter_h, filter_w, stride, pad)

    # 只在高和宽两个维度补 0
    img = np.pad(
        input_data,
        [(0, 0), (0, 0), (pad, pad), (pad, pad)],
        mode="constant"
    )

    # 用来收集所有卷积窗口中的元素
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w), dtype=input_data.dtype)

    # 遍历卷积核内部的每一个位置
    for y in range(filter_h):
        y_max = y + stride * out_h
        for x in range(filter_w):
            x_max = x + stride * out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    # 调整维度顺序，然后把每个窗口拉平成一行
    col = col.transpose(0, 4, 5, 1, 2, 3)
    col = col.reshape(N * out_h * out_w, -1)

    return col


class Convolution:
    def __init__(self, w, b, stride=1, pad=0):
        """
        w shape: (FN, C, FH, FW)
        b shape: (FN,)
        """
        self.w = w
        self.b = b
        self.stride = stride
        self.pad = pad

    def forward(self, x):
        """
        x shape: (N, C, H, W)
        return shape: (N, FN, out_h, out_w)
        """
        N, C, H, W = x.shape
        FN, C_w, FH, FW = self.w.shape

        assert C == C_w, "Input channel must match filter channel"

        out_h, out_w = calc_conv_output_size(H, W, FH, FW, self.stride, self.pad)

        # 1. 输入窗口展开: (N*out_h*out_w, C*FH*FW)
        x_col = im2col(x, FH, FW, self.stride, self.pad)

        # 2. 把每个卷积核拉平成一行，组成权重矩阵
        w_col = self.w.reshape(FN, -1)

        # 3. 矩阵乘法: (N*out_h*out_w, FN)
        out = x_col @ w_col.T + self.b.reshape(1, FN)

        # 4. 恢复成卷积输出形状: (N, FN, out_h, out_w)
        out = out.reshape(N, out_h, out_w, FN)
        out = out.transpose(0, 3, 1, 2)

        return out
#这是逻辑最清楚的代码了，适合宝宝辅食！

In [4]:
#如果权重/偏置为负数，卷积层输出的数据(out)中的数字可能为负。所以激活函数先处理一波保证下一个池子的输入全为非负。
class ReLU:
    def forward(self, x):
        """
        对 NumPy 数组逐元素做 ReLU
        x 可以是任意形状，如 (N, C, H, W)
        """
        return np.maximum(0, x)


In [5]:
class max_pooling:
    def __init__(self,pool_h,pool_w,stride=1,pad=0):
        self.pool_h=pool_h
        self.pool_w=pool_w
        self.stride=stride
        self.pad=pad
    def forward(self, x):
        """
        x shape: (N, C, H, W)
        return shape: (N, C, out_h, out_w)
        """
        N, C, H, W = x.shape
        out_h, out_w=calc_conv_output_size(H, W, self.pool_h, self.pool_w,self.stride,self.pad)
        col=im2col(x,self.pool_h,self.pool_w,self.stride,self.pad)
        #拉一行
        col=col.reshape(-1,self.pool_h*self.pool_w)
        #maximum
        out=np.max(col,axis=1)
        #恢复
        out = out.reshape(N, out_h, out_w, C)
        out = out.transpose(0, 3, 1, 2)
        return out

穿线

In [6]:
class SimpleCNN:
    def __init__(self):
        # 第一层卷积: 输入3通道 -> 输出4通道
        w1 = np.random.randn(4, 3, 3, 3) * 0.01
        b1 = np.zeros(4)

        # 第二层卷积: 输入4通道 -> 输出8通道
        w2 = np.random.randn(8, 4, 3, 3) * 0.01
        b2 = np.zeros(8)

        self.conv1 = Convolution(w1, b1, stride=1, pad=1)
        self.relu1 = ReLU()
        self.pool1 = max_pooling(pool_h=2, pool_w=2, stride=2)

        self.conv2 = Convolution(w2, b2, stride=1, pad=1)
        self.relu2 = ReLU()
        self.pool2 = max_pooling(pool_h=2, pool_w=2, stride=2)

    def forward(self, x):
        x = self.conv1.forward(x)
        x = self.relu1.forward(x)
        x = self.pool1.forward(x)

        x = self.conv2.forward(x)
        x = self.relu2.forward(x)
        x = self.pool2.forward(x)

        return x

x = np.random.randn(2, 3, 32, 32)   # batch=2, RGB图, 32x32

model = SimpleCNN()
out = model.forward(x)

print(out.shape)

(2, 8, 8, 8)
